# Flow

In [ ]:
✅ Tokenization
✅ Embeddings
✅ Positional Encoding
✅ Self-Attention
✅ Multi-Head Attention
✅ FFN
✅ LayerNorm
✅ Encoder Block
✅ Encoder
✅ Causal / Masked Self-Attention

        ↓ NEXT

⬜ Decoder MHA ---> This NoteBook
⬜ Cross-Attention
⬜ Decoder Block
⬜ Full Transformer
⬜ Teacher Forcing
⬜ Output Projection + Softmax
⬜ Cross-Entropy Loss
⬜ Backpropagation
⬜ Tiny Transformer Training
⬜ Text Generation
⬜ GPT / Decoder-only Architecture
⬜ BPE Tokenization
⬜ RoPE
⬜ RMSNorm
⬜ SwiGLU
⬜ GQA / MQA
⬜ KV Cache
⬜ Tiny GPT
⬜ Fine-tuning
⬜ LoRA
⬜ RAG
⬜ Production LLM System

# Problem stmt

# Constraints

# Input/Output

# Function

In [10]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax.
    """
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def causal_mask(seq_len):
    causal_mask = np.triu(
        np.full((seq_len, seq_len), -np.inf),
        k=1
    )
    return causal_mask

def decoder_multi_head_attention(
    X,
    wq,
    wk,
    wv,
    wo,
    num_heads
):
    # Create QKV from input X
    Q=X@wq
    K=X@wk
    V=X@wv
    # define shape 
    batch_size, seq_length, d_model = Q.shape
    # define d_head
    d_head=d_model//num_heads
    # creation of heads
    Q_heads = Q.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    K_heads = K.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    V_heads = V.reshape(
        batch_size, seq_length, num_heads, d_head
    )
    # transpose
    Q_heads = Q_heads.transpose(0, 2, 1, 3)
    K_heads = K_heads.transpose(0, 2, 1, 3)
    V_heads = V_heads.transpose(0, 2, 1, 3)

    # q and kt dot product
    scores = Q_heads @ K_heads.transpose(0, 1, 3, 2)

    # Scale by sqrt(d_head)
    scores = scores / np.sqrt(d_head)

    # Causal mask
    causal = causal_mask(seq_length)

    # Apply mask
    masked = scores + causal

    #Soft masx
    attention_weights = softmax(masked, axis=-1)
    # weigthed sum of v
    head_outputs = attention_weights @ V_heads
    # transpose
    head_outputs = head_outputs.transpose(0, 2, 1, 3)
    #concate
    head_outputs = head_outputs.reshape(
        batch_size, seq_length, d_model
    )

    # Output projection

    output = head_outputs @ wo

    return output,attention_weights

# code run

In [11]:
import numpy as np
import math

X = np.array([[
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0]
]])

Wq = np.eye(4)
Wk = np.eye(4)
Wv = np.eye(4)
Wo = np.eye(4)

num_heads = 2

output, attention_weights = decoder_multi_head_attention(
    X, Wq, Wk, Wv, Wo, num_heads
)

print("Output shape:", output.shape)
print("Attention shape:", attention_weights.shape)
print(attention_weights)

Output shape: (1, 3, 4)
Attention shape: (1, 2, 3, 3)
[[[[1.         0.         0.        ]
   [0.33023845 0.66976155 0.        ]
   [0.33333333 0.33333333 0.33333333]]

  [[1.         0.         0.        ]
   [0.5        0.5        0.        ]
   [0.24825508 0.24825508 0.50348984]]]]
